# Notebook 11 — Benchmark: Fine-tuned TrOCR vs Our CRNN+Lexicon

In [1]:
pip install transformers


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install "transformers==4.40.2"

  Using cached transformers-4.40.2-py3-none-any.whl.metadata (137 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.19.1.tar.gz (321 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.40.2-py3-none-any.whl (9.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [112 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cpython-313-darwin
      Computed rustc target triple: aarch64-apple-darwin
      Installation directory: /Users/avishkashenan/Library/Caches/puccinialin
      Rustup already downloaded
      Installing rust to /Users/avishkashenan/Library/Caches/puccinialin/

In [3]:
pip install sentencepiece tiktoken protobuf


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import random, time
from dataclasses import dataclass
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE} | torch {torch.__version__}")

Using device: mps | torch 2.12.0


In [5]:
@dataclass
class Config:
    data_root: Path = Path("../data/pharmacy_lk")
    train_csv: str = "splits/train.csv"; val_csv: str = "splits/val.csv"; test_csv: str = "splits/test.csv"
    img_dir: str = "images"; img_col: str = "image_filename"; label_col: str = "medicine_name"
    model_name: str = "microsoft/trocr-small-handwritten"   # small variant = feasible on M4
    batch_size: int = 8           # lower to 4 if MPS runs out of memory
    epochs: int = 8               # TrOCR fine-tunes fast (pretrained); few epochs needed
    lr: float = 5e-5
    max_target_len: int = 24
    num_workers: int = 0
    ckpt_dir: Path = Path("../checkpoints/benchmark_trocr")

CFG = Config()
CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)

## 1. Processor + model
TrOCR couples a ViT image encoder with a text decoder. The processor handles image
normalisation and tokenisation. We load the pretrained weights and fine-tune end-to-end.

In [10]:
processor = TrOCRProcessor.from_pretrained(CFG.model_name)
model = VisionEncoderDecoderModel.from_pretrained(CFG.model_name).to(DEVICE)

# decoding/config tokens
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
#model.config.max_length = CFG.max_target_len
print(f"TrOCR loaded: {CFG.model_name} | params={sum(p.numel() for p in model.parameters())/1e6:.0f}M")

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TrOCR loaded: microsoft/trocr-small-handwritten | params=62M


## 2. Dataset (TrOCR expects RGB images + tokenised labels)

In [11]:
class TrOCRDataset(Dataset):
    def __init__(self, csv_path, img_dir, cfg, processor):
        self.df = pd.read_csv(csv_path).dropna(subset=[cfg.label_col]).reset_index(drop=True)
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip()
        self.img_dir = Path(img_dir); self.cfg = cfg; self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / str(row[self.cfg.img_col])).convert("RGB")
        pixel_values = self.processor(img, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            row[self.cfg.label_col], padding="max_length",
            max_length=self.cfg.max_target_len, truncation=True).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels),
                "text": row[self.cfg.label_col], "fname": str(row[self.cfg.img_col])}

def collate(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "text": [b["text"] for b in batch],
        "fname": [b["fname"] for b in batch],
    }

train_ds = TrOCRDataset(CFG.data_root / CFG.train_csv, CFG.data_root / CFG.img_dir, CFG, processor)
val_ds = TrOCRDataset(CFG.data_root / CFG.val_csv, CFG.data_root / CFG.img_dir, CFG, processor)
test_ds = TrOCRDataset(CFG.data_root / CFG.test_csv, CFG.data_root / CFG.img_dir, CFG, processor)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

train_dl = DataLoader(train_ds, CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, collate_fn=collate)
val_dl = DataLoader(val_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)

train=3688 val=790 test=791


## 3. Metrics (same CER / ExactMatch as every other notebook)

In [12]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

def corpus_metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs); exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot / max(chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}

## 4. Fine-tune

In [13]:
@torch.no_grad()
def evaluate(loader):
    model.eval(); preds, refs, files = [], [], []
    for batch in loader:
        gen = model.generate(batch["pixel_values"].to(DEVICE), max_new_tokens=CFG.max_target_len)
        out = processor.batch_decode(gen, skip_special_tokens=True)
        preds += [o.strip().lower() for o in out]
        refs += [t.lower() for t in batch["text"]]; files += batch["fname"]
    return corpus_metrics(preds, refs), preds, refs, files

opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
best_cer = float("inf")
for epoch in range(1, CFG.epochs + 1):
    model.train(); t0, running = time.time(), 0.0
    for k, batch in enumerate(train_dl):
        out = model(pixel_values=batch["pixel_values"].to(DEVICE),
                    labels=batch["labels"].to(DEVICE))
        loss = out.loss
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
        if k % 20 == 0:
            print(f"  epoch {epoch} step {k}/{len(train_dl)} | loss {loss.item():.3f}", end="\r")
    vm, _, _, _ = evaluate(val_dl)
    print(f"\nepoch {epoch} | train loss {running/len(train_dl):.4f} | "
          f"val CER {vm['CER']:.4f} | val EM {vm['ExactMatch']:.4f} | {(time.time()-t0)/60:.1f} min")
    if vm["CER"] < best_cer:
        best_cer = vm["CER"]
        model.save_pretrained(CFG.ckpt_dir / "best")
        print(f"  saved best (val CER {best_cer:.4f})")

  epoch 1 step 460/461 | loss 1.132
epoch 1 | train loss 2.3938 | val CER 0.2918 | val EM 0.3949 | 3.0 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.2918)
  epoch 2 step 460/461 | loss 1.718
epoch 2 | train loss 1.0896 | val CER 0.2753 | val EM 0.4570 | 2.9 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.2753)
  epoch 3 step 460/461 | loss 1.123
epoch 3 | train loss 0.8327 | val CER 0.2203 | val EM 0.5354 | 3.0 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.2203)
  epoch 4 step 460/461 | loss 0.536
epoch 4 | train loss 0.6519 | val CER 0.2447 | val EM 0.5190 | 2.9 min
  epoch 5 step 460/461 | loss 1.313
epoch 5 | train loss 0.5730 | val CER 0.2368 | val EM 0.5443 | 2.9 min
  epoch 6 step 460/461 | loss 0.994
epoch 6 | train loss 0.5384 | val CER 0.2576 | val EM 0.5304 | 2.9 min
  epoch 7 step 460/461 | loss 0.221
epoch 7 | train loss 0.4228 | val CER 0.1996 | val EM 0.5772 | 3.0 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.1996)
  epoch 8 step 460/461 | loss 0.589
epoch 8 | train loss 0.3590 | val CER 0.2104 | val EM 0.5962 | 3.0 min


## 5. Test evaluation + comparison to our champion (M2)

In [14]:
model = VisionEncoderDecoderModel.from_pretrained(CFG.ckpt_dir / "best").to(DEVICE)
tm, preds, refs, files = evaluate(test_dl)
print(f"TrOCR (fine-tuned) — test (n={tm['n']}): CER {tm['CER']:.4f} | EM {tm['ExactMatch']:.4f}")

# seen/unseen breakdown (same protocol)
test_meta = pd.read_csv(CFG.data_root / CFG.test_csv)
seen_map = dict(zip(test_meta[CFG.img_col].astype(str), test_meta["seen_in_train"]))
groups = {"seen": ([], []), "unseen": ([], [])}
for p, r, f in zip(preds, refs, files):
    k = "seen" if seen_map.get(f, False) else "unseen"
    groups[k][0].append(p); groups[k][1].append(r)
print("TrOCR seen/unseen:")
for kk, (P, R) in groups.items():
    if R:
        m = corpus_metrics(P, R)
        print(f"  {kk:6s} (n={m['n']:3d}): CER {m['CER']:.4f} | EM {m['ExactMatch']:.4f}")

pd.DataFrame({"model": ["TrOCR-finetuned"], "CER": [tm["CER"]], "EM": [tm["ExactMatch"]],
              "params_M": [sum(p.numel() for p in model.parameters())/1e6]}
             ).to_csv(CFG.ckpt_dir / "trocr_results.csv", index=False)

print("""
COMPARISON (fill from your runs):
  M0 CRNN baseline (multi-seed) : EM 0.16 ± 0.06
  M2 CRNN + lexicon (CHAMPION)  : EM 0.32 ± 0.08   <- compare TrOCR against this
  TrOCR fine-tuned              : EM (above)
""")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

TrOCR (fine-tuned) — test (n=791): CER 0.2159 | EM 0.5689
TrOCR seen/unseen:
  seen   (n=615): CER 0.1568 | EM 0.7008
  unseen (n=176): CER 0.3989 | EM 0.1080

COMPARISON (fill from your runs):
  M0 CRNN baseline (multi-seed) : EM 0.16 ± 0.06
  M2 CRNN + lexicon (CHAMPION)  : EM 0.32 ± 0.08   <- compare TrOCR against this
  TrOCR fine-tuned              : EM (above)



## 6. For the thesis
- Report TrOCR's params (~330x our CRNN) and its test EM vs our M2.
- If TrOCR ≤ M2: strong finding — "a large pretrained transformer, fine-tuned on identical
  data, does not surpass our lightweight CRNN+lexicon on this small domain dataset", supporting
  the thesis that domain knowledge > raw scale when data is scarce.
- If TrOCR > M2: report honestly; note the compute cost (size, training time) as a practical
  trade-off, and that the lexicon could also be applied on top of TrOCR (future work).
- Either way, also report training time/compute — a fair benchmark includes cost, not just accuracy.